# 02 — Data Preprocessing (Tumor vs No-Tumor)

Preprocessing pipeline for liver CT images used in **Stage 1** (tumor presence detection).

## Pipeline Steps
1. **Load** grayscale CT image
2. **Denoise** with Gaussian blur
3. **Segment** body region using Otsu thresholding
4. **Extract ROI** via largest contour + bounding box with padding
5. **Morphological closing** to fill small holes in the mask
6. **Resize** to 224x224 (bicubic interpolation)
7. **Normalize** to ImageNet mean/std
8. **Augment** both classes to balance at 300 images per class

> **Note:** This notebook processes raw images and saves to `data/train_presence/`.  
> Since data is already preprocessed, **re-running is not required**.

## Configuration

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path

# --- Paths ---
INPUT_NO_TUMOR  = "../data/train_presence/no_tumor"
INPUT_TUMOR     = "../data/train_presence/tumor"
OUTPUT_NO_TUMOR = "../data/train_presence/no_tumor"
OUTPUT_TUMOR    = "../data/train_presence/tumor"

# --- Preprocessing ---
IMG_SIZE       = 224
IMAGENET_MEAN  = np.array([0.485, 0.456, 0.406])
IMAGENET_STD   = np.array([0.229, 0.224, 0.225])
BLUR_KERNEL    = (5, 5)
CONTOUR_PAD    = 5
MORPH_KERNEL   = np.ones((3, 3), np.uint8)

# --- Augmentation ---
TARGET_PER_CLASS = 300
ROTATION_RANGE   = 25
BRIGHTNESS_RANGE = (0.7, 1.3)
IMAGE_EXTS       = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

## Preprocessing Functions

Same ROI extraction pipeline as notebook 01.

In [ ]:
def process_image(img_path: str) -> np.ndarray | None:
    """
    Load a grayscale CT image, extract ROI, resize, and normalize.
    
    Returns:
        Normalized RGB image (H, W, 3) with ImageNet mean/std,
        or None if the image cannot be read.
    """
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"[WARN] Cannot read image: {img_path}")
        return None

    img = np.nan_to_num(img, nan=0, posinf=255, neginf=0).astype(np.uint8)
    blurred = cv2.GaussianBlur(img, BLUR_KERNEL, 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        roi = img
    else:
        largest = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest)

        mask = np.zeros_like(img)
        cv2.drawContours(mask, [largest], -1, 255, -1)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, MORPH_KERNEL, iterations=1)
        masked = cv2.bitwise_and(img, img, mask=mask)

        y0 = max(0, y - CONTOUR_PAD)
        y1 = min(img.shape[0], y + h + CONTOUR_PAD)
        x0 = max(0, x - CONTOUR_PAD)
        x1 = min(img.shape[1], x + w + CONTOUR_PAD)
        roi = masked[y0:y1, x0:x1]

        if roi.size == 0:
            roi = img

    roi = cv2.resize(roi, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
    rgb = cv2.cvtColor(roi, cv2.COLOR_GRAY2RGB)
    normalized = (rgb.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
    return normalized


def denormalize_to_uint8(img: np.ndarray) -> np.ndarray:
    """Convert ImageNet-normalized image back to uint8 RGB for saving."""
    return ((img * IMAGENET_STD + IMAGENET_MEAN) * 255.0).clip(0, 255).astype(np.uint8)


def list_images(folder: str) -> list[str]:
    """List all image files in a directory."""
    return sorted([f for f in os.listdir(folder) if f.lower().endswith(IMAGE_EXTS)])

## Augmentation & Directory Processing

In [ ]:
def augment_image(img: np.ndarray) -> np.ndarray:
    """
    Apply random augmentation to a uint8 RGB image.
    Augmentations: rotation, horizontal flip, brightness shift.
    """
    h, w = img.shape[:2]

    angle = np.random.uniform(-ROTATION_RANGE, ROTATION_RANGE)
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)

    if np.random.random() > 0.5:
        img = cv2.flip(img, 1)

    factor = np.random.uniform(*BRIGHTNESS_RANGE)
    img = np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)

    return img


def process_directory(input_dir: str, output_dir: str, target_count: int):
    """
    Process all images in input_dir and save to output_dir.
    If the class has fewer images than target_count, augment to fill the gap.
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    images = list_images(input_dir)
    print(f"[INFO] Processing: {input_dir}")
    print(f"[INFO] Found {len(images)} images")

    # Step 1: Preprocess and save
    processed = 0
    for filename in images:
        out_path = os.path.join(output_dir, filename)
        if os.path.exists(out_path):
            processed += 1
            continue

        result = process_image(os.path.join(input_dir, filename))
        if result is None:
            continue

        rgb_uint8 = denormalize_to_uint8(result)
        cv2.imwrite(out_path, cv2.cvtColor(rgb_uint8, cv2.COLOR_RGB2BGR))
        processed += 1

    print(f"[INFO] Preprocessed {processed}/{len(images)} images")

    # Step 2: Augment if needed
    current = len(list_images(output_dir))
    need = max(0, target_count - current)

    if need == 0:
        print(f"[INFO] Already at target ({current} images). Skipping augmentation.")
        return

    print(f"[INFO] Need {need} augmented images to reach target ({target_count})")
    source_files = list_images(output_dir)
    created = 0
    idx = 0

    while created < need:
        src_name = source_files[idx % len(source_files)]
        src_img = cv2.imread(os.path.join(output_dir, src_name), cv2.IMREAD_COLOR)
        if src_img is None:
            idx += 1
            continue

        src_rgb = cv2.cvtColor(src_img, cv2.COLOR_BGR2RGB)
        aug_rgb = augment_image(src_rgb)
        aug_bgr = cv2.cvtColor(aug_rgb, cv2.COLOR_RGB2BGR)

        aug_name = f"aug_{created:04d}_{src_name}"
        cv2.imwrite(os.path.join(output_dir, aug_name), aug_bgr)
        created += 1
        idx += 1

        if created % 20 == 0 or created == need:
            print(f"[INFO] Augmentation progress: {created}/{need}")

    print(f"[DONE] Final count: {len(list_images(output_dir))} images in {output_dir}")

## Execute

Process No-Tumor and Tumor classes for Stage 1 training.

In [ ]:
print("=" * 60)
print("Processing NO-TUMOR class")
print("=" * 60)
process_directory(INPUT_NO_TUMOR, OUTPUT_NO_TUMOR, target_count=TARGET_PER_CLASS)

print()

print("=" * 60)
print("Processing TUMOR class")
print("=" * 60)
process_directory(INPUT_TUMOR, OUTPUT_TUMOR, target_count=TARGET_PER_CLASS)

print()
print("=" * 60)
print("Preprocessing complete.")
print(f"  No-Tumor: {len(list_images(OUTPUT_NO_TUMOR))} images")
print(f"  Tumor:    {len(list_images(OUTPUT_TUMOR))} images")
print("=" * 60)